## 검색 결과 융합(RAG Fusion)과 응답 품질 향상

In [1]:
import os
from dotenv import load_dotenv

# .env 파일의 내용 불러오기
load_dotenv("C:/env/.env")

True

In [2]:
# 🔸 LangChain v1.0 기준 Fusion RAG + Answer Refine Agent 예제
# pip install langchain langchain-openai langchain-community faiss-cpu python-dotenv

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain.tools import tool
from langchain.agents import create_agent

# load_dotenv()

# -------------------------------------------------------
# 1) LLM 및 임베딩 모델 초기화
# -------------------------------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# -------------------------------------------------------
# 2) 샘플 문서 및 벡터스토어 초기화
# -------------------------------------------------------
docs = [
    Document(page_content="인공지능은 인간의 학습 능력과 추론을 모방하는 기술이다."),
    Document(page_content="머신러닝은 데이터를 이용해 스스로 패턴을 학습하는 인공지능의 한 분야이다."),
    Document(page_content="RAG는 검색과 생성 모델을 결합해 응답의 정확성을 높인다."),
    Document(page_content="Fusion RAG는 여러 검색 결과를 결합해 응답 품질을 높이는 방법이다."),
]
vectorstore = FAISS.from_documents(docs, embeddings)

# -------------------------------------------------------
# 3) Fusion RAG 도구 정의
# -------------------------------------------------------
@tool
def fusion_rag_search(query: str) -> str:
    """Fusion RAG 방식으로 여러 쿼리 검색 결과를 융합하여 문맥을 반환한다."""
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    sub_queries = [
        query,
        f"{query} 관련 개념을 설명하라",
        f"{query}의 작동 원리를 요약하라",
    ]

    all_results = []
    for q in sub_queries:
        results = retriever.invoke(q)
        all_results.extend(results)

    # 중복 제거
    unique_texts = list({d.page_content for d in all_results})
    fused_context = "\n".join(unique_texts)

    return f"[융합 문맥]\n{fused_context}"

# -------------------------------------------------------
# 4) 응답 품질 향상 도구 정의
# -------------------------------------------------------
@tool
def answer_refiner(context: str, question: str) -> str:
    """Fusion된 문맥을 기반으로 고품질 응답을 생성한다."""
    prompt = f"""
    아래 문맥을 참고하여 질문에 대해 명확하고 근거 있는 답변을 작성하라.

    [문맥]
    {context}

    [질문]
    {question}

    답변:
    """
    response = llm.invoke(prompt)
    return response.content

# -------------------------------------------------------
# 5) 에이전트 생성 (LangChain v1.0 형식)
# -------------------------------------------------------
agent = create_agent(
    model=llm,
    tools=[fusion_rag_search, answer_refiner],
    system_prompt="사용자의 요청을 해결하기 위해 필요시 Fusion RAG과 품질 향상 도구를 사용하라."
)

# -------------------------------------------------------
# 6) 실행: v1에서는 messages 리스트로 전달
# -------------------------------------------------------
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "RAG Fusion이 무엇이며 응답 품질을 어떻게 향상시키는가?"}
    ]
})

print("\n=== 최종 응답 ===")
print(result["messages"][-1].content)


=== 최종 응답 ===
RAG Fusion은 Retrieval-Augmented Generation의 약자로, 정보 검색과 생성 모델을 결합하여 응답의 품질을 향상시키는 방법입니다. 이 접근 방식은 먼저 관련 정보를 검색한 후, 이 정보를 바탕으로 고품질의 응답을 생성하는 과정을 포함합니다.

응답 품질을 향상시키는 방법은 다음과 같습니다:

1. **정보 검색**: RAG Fusion은 다양한 출처에서 정보를 검색하여 관련 데이터를 수집합니다. 이 과정에서 여러 검색 결과를 활용하여 보다 풍부한 문맥을 형성합니다.

2. **정보 융합**: 수집된 여러 검색 결과를 통합하여 문맥을 형성함으로써, 단일 출처에 의존하지 않고 다양한 관점을 반영할 수 있습니다. 이는 응답의 신뢰성을 높이는 데 기여합니다.

3. **정확성과 관련성**: 검색된 정보는 생성 모델에 입력되어, 보다 정확하고 관련성 높은 응답을 생성하는 데 사용됩니다. 이로 인해 사용자가 원하는 정보에 보다 적합한 답변을 제공할 수 있습니다.

결과적으로, RAG Fusion은 정보의 다양성과 정확성을 결합하여, 사용자에게 더 나은 품질의 응답을 제공하는 효과적인 방법입니다.


### RAG Fusion활용 예제

In [3]:
"""
🔸 LangChain v1.0 기준 Fusion RAG + Answer Refine Agent 예제
검색 결과 융합(RAG Fusion)과 응답 품질 향상 에이전트 구현

주요 기능:
1. Fusion RAG: 여러 쿼리로 검색하여 결과를 융합
2. Answer Refine Agent: 융합된 문맥을 기반으로 고품질 응답 생성
3. LangChain v1.0 에이전트 프레임워크 활용
"""

import os
from typing import List, Dict, Any
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage

# 환경 변수 로드
# load_dotenv()

class FusionRAGAgent:
    """Fusion RAG와 Answer Refine Agent를 결합한 클래스"""
    
    def __init__(self, model_name: str = "gpt-4o-mini", embedding_model: str = "text-embedding-3-small"):
        """
        Fusion RAG Agent 초기화
        
        Args:
            model_name: 사용할 LLM 모델명
            embedding_model: 사용할 임베딩 모델명
        """
        self.llm = ChatOpenAI(model=model_name, temperature=0.2)
        self.embeddings = OpenAIEmbeddings(model=embedding_model)
        self.vectorstore = None
        self.agent = None
        
    def setup_knowledge_base(self, documents: List[Document]):
        """
        지식 베이스 설정
        
        Args:
            documents: 벡터스토어에 저장할 문서 리스트
        """
        print("📚 지식 베이스 구축 중...")
        self.vectorstore = FAISS.from_documents(documents, self.embeddings)
        print(f"✅ {len(documents)}개 문서가 벡터스토어에 저장되었습니다.")
        
    def create_fusion_rag_tool(self):
        """Fusion RAG 검색 도구 생성"""
        
        @tool
        def fusion_rag_search(query: str) -> str:
            """
            Fusion RAG 방식으로 여러 쿼리 검색 결과를 융합하여 문맥을 반환한다.
            
            Args:
                query: 검색할 질문
                
            Returns:
                융합된 문맥 정보
            """
            if not self.vectorstore:
                return "벡터스토어가 초기화되지 않았습니다."
                
            retriever = self.vectorstore.as_retriever(search_kwargs={"k": 3})
            
            # 다양한 관점의 서브 쿼리 생성
            sub_queries = [
                query,
                f"{query} 관련 개념을 설명하라",
                f"{query}의 작동 원리를 요약하라",
                f"{query}의 장단점은 무엇인가",
                f"{query}의 실제 활용 사례는 무엇인가"
            ]
            
            all_results = []
            for q in sub_queries:
                try:
                    results = retriever.invoke(q)
                    all_results.extend(results)
                except Exception as e:
                    print(f"쿼리 '{q}' 검색 중 오류: {e}")
                    continue
            
            # 중복 제거 및 문맥 융합
            unique_texts = list({d.page_content for d in all_results})
            fused_context = "\n".join(unique_texts)
            
            return f"[융합 문맥]\n{fused_context}\n\n[검색된 문서 수: {len(unique_texts)}개]"
        
        return fusion_rag_search
    
    def create_answer_refiner_tool(self):
        """응답 품질 향상 도구 생성"""
        
        @tool
        def answer_refiner(context: str, question: str) -> str:
            """
            Fusion된 문맥을 기반으로 고품질 응답을 생성한다.
            
            Args:
                context: 융합된 문맥 정보
                question: 원본 질문
                
            Returns:
                개선된 응답
            """
            prompt = f"""
            아래 문맥을 참고하여 질문에 대해 명확하고 근거 있는 답변을 작성하라.
            
            답변 작성 시 다음 사항을 고려하라:
            1. 문맥의 정보를 충분히 활용하라
            2. 구체적이고 실용적인 예시를 포함하라
            3. 명확한 구조로 답변을 구성하라
            4. 불확실한 정보는 명시하라
            
            [문맥]
            {context}
            
            [질문]
            {question}
            
            답변:
            """
            
            try:
                response = self.llm.invoke(prompt)
                return response.content
            except Exception as e:
                return f"응답 생성 중 오류가 발생했습니다: {e}"
        
        return answer_refiner
    
    def create_agent(self):
        """Fusion RAG 에이전트 생성"""
        
        # 도구 생성
        fusion_rag_tool = self.create_fusion_rag_tool()
        answer_refiner_tool = self.create_answer_refiner_tool()
        
        # 시스템 프롬프트 정의
        system_prompt = """
        당신은 Fusion RAG와 Answer Refine Agent를 활용하는 지능형 질의응답 시스템입니다.
        
        작업 순서:
        1. 사용자의 질문을 분석한다
        2. fusion_rag_search 도구를 사용해 관련 문맥을 검색하고 융합한다
        3. answer_refiner 도구를 사용해 융합된 문맥을 기반으로 고품질 응답을 생성한다
        4. 최종 응답을 사용자에게 제공한다
        
        항상 근거 있는 답변을 제공하고, 불확실한 정보는 명시하라.
        """
        
        # 에이전트 생성
        self.agent = create_agent(
            model=self.llm,
            tools=[fusion_rag_tool, answer_refiner_tool],
            system_prompt=system_prompt
        )
        
        print("🤖 Fusion RAG 에이전트가 생성되었습니다.")
    
    def query(self, question: str) -> str:
        """
        질문에 대한 응답 생성
        
        Args:
            question: 사용자 질문
            
        Returns:
            에이전트의 응답
        """
        if not self.agent:
            return "에이전트가 초기화되지 않았습니다. create_agent()를 먼저 호출하세요."
        
        try:
            result = self.agent.invoke({
                "messages": [
                    HumanMessage(content=question)
                ]
            })
            
            # 마지막 AI 메시지의 내용 반환
            for message in reversed(result["messages"]):
                if isinstance(message, AIMessage):
                    return message.content
            
            return "응답을 생성할 수 없습니다."
            
        except Exception as e:
            return f"질의응답 처리 중 오류가 발생했습니다: {e}"


def create_sample_documents() -> List[Document]:
    """샘플 문서 생성"""
    return [
        Document(page_content="인공지능(AI)은 인간의 학습 능력과 추론을 모방하는 기술이다. 머신러닝, 딥러닝, 자연어처리 등의 하위 분야를 포함한다."),
        Document(page_content="머신러닝은 데이터를 이용해 스스로 패턴을 학습하는 인공지능의 한 분야이다. 지도학습, 비지도학습, 강화학습으로 분류된다."),
        Document(page_content="RAG(Retrieval-Augmented Generation)는 검색과 생성 모델을 결합해 응답의 정확성과 신뢰성을 높이는 기술이다."),
        Document(page_content="Fusion RAG는 여러 검색 결과를 결합해 응답 품질을 높이는 방법이다. 다양한 관점의 쿼리를 통해 더 포괄적인 정보를 수집한다."),
        Document(page_content="벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템이다. FAISS, Pinecone, Weaviate 등이 대표적이다."),
        Document(page_content="임베딩은 텍스트를 고차원 벡터로 변환하는 기술이다. 의미적 유사성을 벡터 공간에서 측정할 수 있게 해준다."),
        Document(page_content="LangChain은 LLM 애플리케이션 개발을 위한 프레임워크이다. 체인, 에이전트, 메모리 등의 기능을 제공한다."),
        Document(page_content="에이전트는 도구를 사용해 복잡한 작업을 수행하는 AI 시스템이다. 계획 수립, 도구 선택, 실행, 평가의 사이클을 반복한다."),
    ]


def main():
    """메인 실행 함수"""
    print("🚀 Fusion RAG + Answer Refine Agent 예제 시작")
    print("=" * 50)
    
    # 1. Fusion RAG Agent 초기화
    agent = FusionRAGAgent()
    
    # 2. 샘플 문서로 지식 베이스 구축
    documents = create_sample_documents()
    agent.setup_knowledge_base(documents)
    
    # 3. 에이전트 생성
    agent.create_agent()
    
    # 4. 테스트 질문들
    test_questions = [
        "RAG Fusion이 무엇이며 응답 품질을 어떻게 향상시키는가?",
        "머신러닝과 딥러닝의 차이점은 무엇인가?",
        "벡터 데이터베이스의 역할과 장점은 무엇인가?",
        "LangChain 에이전트의 작동 원리를 설명하라"
    ]
    
    print("\n📝 테스트 질문 실행")
    print("=" * 50)
    
    for i, question in enumerate(test_questions, 1):
        print(f"\n🔍 질문 {i}: {question}")
        print("-" * 30)
        
        response = agent.query(question)
        print(f"💡 응답: {response}")
        print("-" * 50)
    
    # 5. 대화형 모드
    print("\n💬 대화형 모드 (종료하려면 'quit' 입력)")
    print("=" * 50)
    
    while True:
        try:
            user_input = input("\n질문을 입력하세요: ").strip()
            
            if user_input.lower() in ['quit', 'exit', '종료']:
                print("👋 프로그램을 종료합니다.")
                break
            
            if not user_input:
                continue
                
            print("\n🤔 답변 생성 중...")
            response = agent.query(user_input)
            print(f"\n💡 응답: {response}")
            
        except KeyboardInterrupt:
            print("\n👋 프로그램을 종료합니다.")
            break
        except Exception as e:
            print(f"\n❌ 오류가 발생했습니다: {e}")


if __name__ == "__main__":
    main()

# 질문 예시
# 랭체인이 뭐지?
# RAG의 기능은?
# 임베딩은?

🚀 Fusion RAG + Answer Refine Agent 예제 시작
📚 지식 베이스 구축 중...
✅ 8개 문서가 벡터스토어에 저장되었습니다.
🤖 Fusion RAG 에이전트가 생성되었습니다.

📝 테스트 질문 실행

🔍 질문 1: RAG Fusion이 무엇이며 응답 품질을 어떻게 향상시키는가?
------------------------------
💡 응답: RAG Fusion은 Retrieval-Augmented Generation의 한 형태로, 여러 검색 결과를 통합하여 보다 높은 품질의 응답을 생성하는 방법입니다. 이 기술은 검색과 생성 모델을 결합하여 사용자가 요청한 정보에 대해 보다 정확하고 신뢰할 수 있는 답변을 제공하는 데 중점을 둡니다.

### RAG Fusion의 구성 요소
1. **검색 모델**: 사용자의 쿼리에 대해 다양한 소스에서 정보를 검색합니다. 예를 들어, 사용자가 "기후 변화의 영향"에 대해 질문하면, 검색 모델은 관련된 논문, 기사, 블로그 포스트 등을 찾아냅니다.
   
2. **생성 모델**: 검색된 정보를 바탕으로 자연어로 응답을 생성합니다. 이 과정에서 생성 모델은 여러 관점의 정보를 통합하여 일관성 있고 포괄적인 답변을 만들어냅니다.

### 응답 품질 향상 방법
1. **다양한 관점 수집**: RAG Fusion은 여러 검색 결과를 결합함으로써 다양한 관점의 정보를 수집합니다. 예를 들어, 기후 변화에 대한 과학적 데이터, 정책적 논의, 개인적인 경험 등을 모두 포함할 수 있습니다. 이로 인해 사용자는 보다 균형 잡힌 정보를 얻을 수 있습니다.

2. **정확성과 신뢰성 증대**: 검색된 정보의 신뢰성을 평가하고, 이를 바탕으로 생성 모델이 응답을 작성하기 때문에, 결과적으로 더 정확하고 신뢰할 수 있는 답변이 제공됩니다. 예를 들어, 기후 변화의 영향에 대한 질문에 대해, RAG Fusion은 최신 연구 결과와 전문가 의견을 반영하여 답변을 생성할 수 있습니다.

3. **정보의 포괄성**: 여러 출처에서 정


질문을 입력하세요:  랭체인이 뭐지?



🤔 답변 생성 중...

💡 응답: 랭체인(LangChain)은 대규모 언어 모델(LLM) 애플리케이션 개발을 위한 강력한 프레임워크입니다. 이 프레임워크는 개발자가 LLM을 활용하여 다양한 기능을 구현할 수 있도록 돕는 여러 가지 도구와 구조를 제공합니다.

### 주요 기능
1. **체인(Chain)**: 여러 개의 작업을 순차적으로 연결하여 복잡한 프로세스를 구성할 수 있습니다. 예를 들어, 사용자가 질문을 입력하면, LangChain은 이를 처리하여 관련 정보를 검색하고, 그 결과를 요약하여 사용자에게 제공하는 일련의 작업을 자동으로 수행할 수 있습니다.

2. **에이전트(Agent)**: 에이전트는 특정 작업을 수행하기 위해 외부 API나 데이터베이스와 상호작용할 수 있는 기능을 제공합니다. 예를 들어, 사용자가 특정 주제에 대한 정보를 요청하면, 에이전트는 웹에서 정보를 검색하고, 이를 바탕으로 사용자에게 적절한 답변을 제공할 수 있습니다.

3. **메모리(Memory)**: LangChain은 대화의 맥락을 유지하기 위해 메모리 기능을 제공합니다. 이를 통해 사용자가 이전에 했던 질문이나 대화 내용을 기억하고, 더 자연스러운 대화를 이어갈 수 있습니다. 예를 들어, 사용자가 "지난번에 추천해준 책이 뭐였지?"라고 질문하면, LangChain은 이전 대화 내용을 참조하여 적절한 답변을 제공할 수 있습니다.

### 실용적인 예시
LangChain을 활용한 실제 애플리케이션의 예로는 고객 지원 챗봇이 있습니다. 이 챗봇은 사용자의 질문을 이해하고, 필요한 정보를 검색하여 즉각적으로 답변할 수 있습니다. 또한, 사용자의 이전 대화 내용을 기억하여 보다 개인화된 서비스를 제공할 수 있습니다.

### 결론
결론적으로, LangChain은 LLM을 활용한 애플리케이션 개발을 간소화하고, 다양한 기능을 통해 개발자가 보다 효율적으로 작업할 수 있도록 돕는 프레임워크입니다. 이를 통해 사용자는 보다 직관적이고 유용한 LLM 기반 서비스를 경험할 수 있습니


질문을 입력하세요:  임베딩은?



🤔 답변 생성 중...

💡 응답: 임베딩(Embedding)은 텍스트, 이미지, 또는 기타 데이터 유형을 고차원 벡터로 변환하는 기술입니다. 이 과정은 데이터의 의미적 유사성을 벡터 공간에서 측정할 수 있도록 해줍니다. 예를 들어, "고양이"와 "강아지"라는 단어는 서로 유사한 의미를 가지므로, 이 두 단어의 임베딩 벡터는 서로 가까운 위치에 배치됩니다. 반면에 "고양이"와 "자동차"는 의미적으로 다르기 때문에 이들의 벡터는 멀리 떨어져 있게 됩니다.

임베딩은 자연어 처리(NLP) 및 기계 학습 분야에서 매우 중요한 역할을 합니다. 예를 들어, 문서 검색 시스템에서는 사용자가 입력한 쿼리와 관련된 문서들을 찾기 위해 임베딩을 사용하여 쿼리와 문서 간의 유사성을 계산합니다. 추천 시스템에서도 사용자의 선호도를 벡터로 표현하여 비슷한 취향을 가진 다른 사용자나 아이템을 추천하는 데 활용됩니다.

이러한 고차원 벡터를 효율적으로 저장하고 검색하기 위해 벡터 데이터베이스가 사용됩니다. FAISS, Pinecone, Weaviate와 같은 시스템은 대량의 벡터 데이터를 빠르게 처리하고 검색할 수 있는 기능을 제공합니다. 예를 들어, Pinecone은 실시간으로 벡터를 추가하고 검색할 수 있는 기능을 제공하여, 추천 시스템이나 검색 엔진에서의 응답 속도를 크게 향상시킵니다.

결론적으로, 임베딩은 데이터의 의미를 수치적으로 표현하여 다양한 응용 프로그램에서 유용하게 활용되는 기술입니다. 이 기술은 특히 대량의 데이터에서 유사성을 찾고, 이를 기반으로 한 다양한 서비스 개발에 필수적입니다.



질문을 입력하세요:  quit


👋 프로그램을 종료합니다.
